In [ ]:
# === Notebook path configuration ===
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
if CWD.name == "diagnostic":
    DIAGNOSTIC_DIR = CWD
elif (CWD / "diagnostic").is_dir():
    DIAGNOSTIC_DIR = CWD / "diagnostic"
else:
    DIAGNOSTIC_DIR = next((p for p in [CWD, *CWD.parents] if p.name == "diagnostic"), CWD.parent)

if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

import os
import numpy as np
import warnings 
from string import Template
import xarray as xr
import numpy as np
import pandas as pd

from datetime import datetime

from util.land_atmosphere_observation_data import ( 
     ObsDataReader,
    
)
from util.land_atmosphere_model_data import (
    ModelDataReader,
)

from configs.land_atmosphere_experiment_configs import get_experiment_dict


In [2]:
import os
import numpy as np
import xarray as xr
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import pandas as pd

class TerrestrialCouplingIndexCalculator:
    def __init__(self, data_dir, experiment, ensemble_list, years,
                 et_var='LHFLX', t2m_var='T2M',
                 lat_bounds=None, lon_bounds=None,
                 outdir='./tci_output', landmask_file=None):
        self.data_dir = data_dir
        self.experiment = experiment
        self.ensemble_list = ensemble_list
        self.years = years
        self.et_var = et_var
        self.t2m_var = t2m_var
        self.lat_bounds = lat_bounds
        self.lon_bounds = lon_bounds
        self.outdir = os.path.join(outdir, experiment)
        os.makedirs(self.outdir, exist_ok=True)

        self.landmask = None
        if landmask_file and os.path.exists(landmask_file):
            self.landmask = xr.open_dataset(landmask_file)['landmask']

        self.Lv = 2.5e6  # J/kg
        self.expected_frequency = 'daily'
        
    def _resample_if_needed(self, ds):
        if self.expected_frequency.lower() == '6h':
            print("[INFO] Resampling 6-hourly data to daily means...")
            return ds.resample(time='1D').mean()
        return ds

    def _load_variable(self, varname, ensemble):
        original_var = varname
        possible_vars = [varname]
        if varname == "LHFLX":
            possible_vars = ["LHFLX", "QFLX"]

        alt_dir = self.data_dir.replace("daily", "6hourly")

        for v in possible_vars:
            for base_dir in [self.data_dir, alt_dir]:
                files = [os.path.join(base_dir, f"{v}.{ensemble}.{year}.nc") for year in self.years]
                if all(os.path.exists(f) for f in files):
                    if base_dir == alt_dir:
                        self.expected_frequency = '6h'
                        print(f"[INFO] {v} found in 6-hourly path: resampling will be applied.")
                    datasets = [xr.open_dataset(f)[v] for f in files]
                    ds = xr.concat(datasets, dim='time')
                    ds = self._resample_if_needed(ds)
                    
                    if original_var == "LHFLX" and v == "QFLX":
                        print("[INFO] Converting QFLX to LHFLX using Lv.")
                        ds = ds * self.Lv
                        ds.attrs["units"] = "W/m2"
                        ds.name = "LHFLX"

                    if self.lat_bounds:
                        ds = ds.sel(lat=slice(*self.lat_bounds))
                    if self.lon_bounds:
                        ds = ds.sel(lon=slice(*self.lon_bounds))

                    return ds

        raise FileNotFoundError(f"[ERROR] Could not find {possible_vars} in {self.data_dir} or {alt_dir} for {ensemble} in daily or 6-hourly paths.")

    def _compute_r2_gridwise(self, et, t2m):
        et_stack = et.stack(grid=('lat', 'lon')).transpose('time', 'grid')
        t2m_stack = t2m.stack(grid=('lat', 'lon')).transpose('time', 'grid')

        r2_vals = []
        for i in range(et_stack.shape[1]):
            x = et_stack[:, i].values
            y = t2m_stack[:, i].values
            if np.all(np.isfinite(x)) and np.all(np.isfinite(y)) and x.size > 1:
                r, _ = pearsonr(x, y)
                r2_vals.append(r**2)
            else:
                r2_vals.append(np.nan)

        r2_array = np.array(r2_vals).reshape((et.sizes['lat'], et.sizes['lon']))
        return xr.DataArray(
            r2_array, coords={'lat': et.lat, 'lon': et.lon},
            dims=['lat', 'lon'], name='TCI'
        )

    def compute_weighted_mean(self, tci_all):
        weights = np.cos(np.deg2rad(tci_all['lat']))
        weights.name = "weights"
        if self.landmask is not None:
            landmask_interp = self.landmask.interp_like(tci_all, method='nearest')
            weights = weights * landmask_interp
        weighted = tci_all.weighted(weights)
        return weighted.mean(dim=['lat', 'lon'], skipna=True)

    def save_regional_mean(self, tci_all, year, ens, frequency):
        mean_tci = self.compute_weighted_mean(tci_all)
        ds_out = xr.Dataset({f"TCI_{frequency}": mean_tci})
        outfile = os.path.join(self.outdir, f"TCI_{frequency}_mean_{year}_{ens}.nc")
        ds_out.to_netcdf(outfile)
        print(f"[SAVED] Regional weighted mean: {outfile}")

    def plot_regional_mean(self, ds, varname='TCI_monthly', savepath=None):
        time = ds['time'].values
        tci = ds[varname].values
        plt.figure(figsize=(10, 4))
        plt.plot(time, tci, marker='o', linestyle='-')
        plt.title(f"{varname} Time Series")
        plt.xlabel("Time")
        plt.ylabel("TCI (weighted mean)")
        plt.grid(True)
        if savepath:
            plt.savefig(savepath, dpi=150)
            print(f"[SAVED] Plot: {savepath}")
        else:
            plt.show()

    def process(self, frequency='monthly', compute_regional_mean=False):
        for ens in self.ensemble_list:
            print(f"[INFO] Processing {self.experiment} | {ens}")
            et = self._load_variable(self.et_var, ens)
            t2m = self._load_variable(self.t2m_var, ens)

            if not np.issubdtype(et['time'].dtype, np.datetime64):
                base_time = np.datetime64(f"{self.years[0]}-01-01")
                et['time'] = base_time + xr.to_timedelta(et['time'], unit='h')
                t2m['time'] = et['time']

            group_key = "D" if frequency == 'daily' else "M"
            time_index = pd.to_datetime(et.time.values)
            group_labels = time_index.to_period(group_key).strftime('%Y-%m') if frequency == 'monthly' else time_index.to_period(group_key).strftime('%Y-%m-%d')
            et.coords['group'] = ('time', group_labels)
            t2m.coords['group'] = ('time', group_labels)

            grouped_et = et.groupby('group')
            grouped_t2m = t2m.groupby('group')
            
            all_tci = []
            for (label_et, et_group), (label_t2m, t2m_group) in zip(grouped_et, grouped_t2m):
                if label_et != label_t2m:
                    print(f"[WARN] Mismatched labels: {label_et} vs {label_t2m}")
                    continue
                if et_group.time.size < 2:
                    continue
                tci = self._compute_r2_gridwise(et_group, t2m_group)
                tci['time'] = et_group.time.mean()
                tci = tci.expand_dims('time')
                all_tci.append(tci)

            if not all_tci:
                print(f"[WARN] No valid TCI computed for {ens}")
                continue

            tci_all = xr.concat(all_tci, dim='time')
            year = self.years[0]
            file_out = os.path.join(self.outdir, f"TCI_{frequency}_{year}_{ens}.nc")
            tci_all.to_netcdf(file_out)
            print(f"[SAVED] {file_out}")

            if compute_regional_mean:
                self.save_regional_mean(tci_all, year, ens, frequency)

In [3]:
if __name__ == "__main__":
    from string import Template
    import os

    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"
    landmask_file = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart/landmask_1x1.nc"

    exp_key = {
        "v3_spinup": 2011,
        "v3_hindcast": 2012
    }
    exp = 'v3_spinup'
    exp_dict = get_experiment_dict(exp)
    years = [exp_key[exp]]

    for exp_name, exp_meta in exp_dict.items():
        print(f"\n[INFO] Running TCI for experiment: {exp_name}")
        path = exp_meta['path']
        template_str = exp_meta['template']
        template = Template(template_str.replace('%(', '${').replace(')s', '}'))

        nens = exp_meta['nens']
        ensemble_list = [f"EN{n:02d}" for n in range(1, nens + 1)]

        tci_calc = TerrestrialCouplingIndexCalculator(
            data_dir=path,
            experiment=exp_name,
            ensemble_list=ensemble_list,
            years=years,
            et_var='LHFLX',
            t2m_var='TREFHT',  # use 'TSA' for 2m temperature approximation in E3SM
            lat_bounds=(-90, 90),
            lon_bounds=(0, 360),
            outdir=os.path.join(out_path, 'tci')
        )

        tci_calc.process()



[INFO] Running TCI for experiment: CTRLEN10
[INFO] Processing CTRLEN10 | EN01
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[INFO] Resampling 6-hourly data to daily means...
[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_dart/tci/CTRLEN10/TCI_monthly_2011_EN01.nc
[INFO] Processing CTRLEN10 | EN02
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[INFO] Resampling 6-hourly data to daily means...
[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_dart/tci/CTRLEN10/TCI_monthly_2011_EN02.nc
[INFO] Processing CTRLEN10 | EN03
[INFO] LHFLX found in 6-hourly path: resampling will be applied.
[INFO] Resampling 6-hourly data to daily means...
[INFO] Resampling 6-hourly data to daily means...
[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_dart/tci/CTRLEN10/TCI_monthly_2011_EN03.nc
[INFO] Processing CTRLEN10 | EN04
[INFO] LHFLX found in 6-hourly path: resampli


KeyboardInterrupt

